In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Configura a renderização gráfica inline no Jupyter notebook
%matplotlib inline

# Decodificação entre sujeitos (*cross-subject*) em gravações reais de SSVEP

Um decodificador consegue identificar o estímulo oscilante (*flickering*) ao qual um novo participante prestou atenção?
Carregamos três participantes do conjunto de dados de SSVEP Nakanishi2015 por meio de
:class:`eegdash.EEGDashDataset`, extraímos janelas rotuladas por eventos com o Braindecode
e avaliamos uma linha de base espectral usando validação cruzada deixando um sujeito de fora (*leave-one-subject-out* - LOSO).

**Dados:** ``nm000118``, sujeitos ``1``, ``2``, ``3``, sessão ``0``, execução (*run*) ``0``.
Os três arquivos de sinal totalizam 21.1 MB, além de pequenos metadados auxiliares BIDS. Acesso à internet é
necessário para o primeiro download; defina ``EEGDASH_CACHE_DIR`` para reutilizá-lo em CI.
CPU é suficiente. Instale o EEGDash e suas dependências antes de executar.

Estes são sinais gravados reais distribuídos como um conjunto de dados BIDS processado,
não a aquisição bruta original sem processamento. Consulte o
[conjunto de dados NEMAR](https://nemar.org/dataset/nm000118) e o
[estudo original](https://doi.org/10.1371/journal.pone.0140703). A versão disponibilizada
já inclui filtragem, redução da taxa de amostragem (*downsampling*) e tratamento de latência; utilizamos seus
inícios de eventos (*onsets*) sem adicionar outra correção de latência.

Pré-requisitos são janelas rotuladas por eventos (tutorial 02) e divisão por grupos
(tutorial 11). Este script recarrega seus próprios dados, portanto nenhum CSV de características ou checkpoint
treinado é necessário. Mantemos a definição de características e a regularização
fixas antes do LOSO; usar essas três partições de teste para escolher uma configuração melhor
transformaria as pontuações de teste relatadas em pontuações de seleção de modelo.


## 1. Selecionar uma coorte pequena e explícita
Filtrar sujeitos, sessão e execução delimita o download. Cortar (*crop*) após
abrir uma gravação reduziria a computação, mas não o tamanho do download.



In [ ]:
# Importa módulos do sistema operacional, manipulação de caminhos e funções parciais
import os
from functools import partial
from pathlib import Path

# Importa bibliotecas para plotagem, computação matricial e manipulação de tabelas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# Importa gerador de janelas baseadas em eventos da Braindecode
from braindecode.preprocessing import create_windows_from_events
# Importa regressor logístico, matriz de confusão e métricas do scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score
# Importa divisão Leave-One-Group-Out, pipeline e padronizador de escala
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Importa classes de dataset e extratores de características espectrais do EEGDash
from eegdash import EEGDashDataset
from eegdash.features import (
    FeatureExtractor,
    extract_features,
    spectral_bands_power,
    spectral_preprocessor,
)

# Define o diretório de cache a partir da variável de ambiente ou valor padrão
cache_dir = Path(os.environ.get("EEGDASH_CACHE_DIR", ".eegdash_cache"))
# Lista de sujeitos selecionados para o protocolo inter-sujeito
subjects = ["1", "2", "3"]
# Inicializa e faz o download das gravações da tarefa de SSVEP
dataset = EEGDashDataset(
    cache_dir=cache_dir,
    dataset="nm000118",
    subject=subjects,
    session="0",
    run="0",
    task="ssvep",
    n_jobs=1,
)
# Valida que exatamente três gravações foram carregadas com sucesso
assert len(dataset.datasets) == len(subjects), "Expected one recording per subject"
# Exibe os metadados de identificação do dataset
print(dataset.description[["subject", "session", "run"]])

## 2. Inspecionar anotações reais e verificar o contrato do sinal
Acessar ``raw`` baixa aquela gravação. Nomes de anotação identificam a
frequência do estímulo atendido em Hz; eles fornecem cada rótulo de classificação.
Todos os participantes devem ter a mesma ordem de canais e frequência de amostragem.



In [ ]:
# Acessa a gravação bruta do primeiro participante para inspecionar metadados
raw = dataset.datasets[0].raw
sfreq = raw.info["sfreq"]
channel_names = raw.ch_names
# Extrai e ordena numericamente as frequências de estímulo presentes nas anotações
class_names = sorted(set(raw.annotations.description), key=float)
# Cria o mapeamento entre os nomes das frequências e os índices de classe (0 a 11)
mapping = {name: index for index, name in enumerate(class_names)}
# Valida a presença exata das doze frequências de estímulo SSVEP
assert len(mapping) == 12, "Expected the twelve SSVEP stimulus frequencies"
# Garante que todos os sujeitos compartilham a mesma lista de canais, taxa de amostragem e rótulos
for recording in dataset.datasets:
    recording_raw = recording.raw
    assert recording_raw.ch_names == channel_names
    assert recording_raw.info["sfreq"] == sfreq
    assert set(recording_raw.annotations.description) == set(mapping)
# Imprime configurações básicas de aquisição e frequências identificadas
print(f"Channels: {channel_names}; sampling frequency: {sfreq} Hz")
print("Stimulus frequencies (Hz):", class_names)

## 3. Fazer uma janela de quatro segundos por ensaio anotado
Cada intervalo anotado dura 4.15 segundos. Mantenha seus primeiros quatro segundos e
descarte o restante. Tamanho e passo (*stride*) explícitos evitam janelas sobrepostas
ou a extensão da época além da duração do evento gravado.
A 256 Hz, quatro segundos contêm 1.024 amostras. O array resultante possui
eixos (540 ensaios, 8 canais de EEG, 1.024 amostras), com dados expressos em volts.
Os metadados possuem uma linha por linha do array. ``target`` é um índice de classe, não uma
frequência em Hz; ``mapping`` é a conversão explícita entre eles.
A fonte contém 15 ensaios de cada uma das 12 frequências por pessoa.
Uma classe ausente é uma quebra no contrato de dados, não um motivo para re-rotular ensaios.



In [ ]:
# Define o tamanho da janela em amostras correspondente a 4 segundos (4 * 256 = 1024 amostras)
window_size = int(4 * sfreq)
# Cria as janelas baseadas nas anotações descartando trechos incompletos ao final
windows = create_windows_from_events(
    dataset,
    mapping=mapping,
    trial_start_offset_samples=0,
    trial_stop_offset_samples=0,
    window_size_samples=window_size,
    window_stride_samples=window_size,
    on_last_window="drop",
    preload=True,
)
# Extrai a tabela de metadados das janelas criadas
metadata = windows.get_metadata()
# Obtém o vetor de classes alvo como array numérico de inteiros
y = metadata["target"].to_numpy(dtype=int)
# Obtém os identificadores dos sujeitos como agrupamento de validação
groups = metadata["subject"].astype(str).to_numpy()
# Empilha os tensores de sinal de cada janela em uma matriz numpy 3D
X = np.stack([window[0] for window in windows])
# Valida o formato tridimensional, os sujeitos presentes e a ausência de valores inválidos (NaN/Inf)
assert X.shape == (len(metadata), len(channel_names), window_size)
assert set(groups) == set(subjects)
assert np.isfinite(X).all()
# Exibe tabela cruzada de contagem de ensaios por sujeito e por classe
print(pd.crosstab(groups, y, rownames=["subject"], colnames=["class"]))

## 4. Extrair características espectrais de cada janela
As respostas de SSVEP contêm energia na frequência do estímulo. Use o log da potência espectral
em torno de cada frequência de estímulo, mantendo todos os oito canais posteriores.
Esta transformação por janela não aprende nada de outros ensaios ou sujeitos.
O escalonador abaixo, por outro lado, deve ser ajustado apenas nos sujeitos de treino.
O pré-processador espectral compartilhado do EEGDash calcula uma PSD de Welch com um segmento Hann
de quatro segundos e bins de 0.25 Hz. Cada banda estreita é centrada
em uma frequência de estímulo documentada; esses centros definem a tarefa, não
preditores específicos de ensaios. Reter oito canais fornece 12 × 8 = 96
características. A versão já processada de SSVEP não precisa de outro passe de limpeza
do EEGPrep ou correção de latência visual.

``spectral_bands_power`` soma os bins selecionados da PSD. Multiplicar pelo espaçamento
de 0.25 Hz converte V²/Hz para a potência aproximada da banda em V². O log comprime
essa escala; o StandardScaler ainda é ajustado apenas nos participantes de treino.



In [ ]:
# Define bandas estreitas de +/- 0.125 Hz ao redor de cada frequência central de estímulo
bands = {
    f"hz_{name}": (float(name) - 0.125, float(name) + 0.125) for name in class_names
}
# Configura extrator espectral com pré-processador PSD Welch de 4 a 16 Hz e segmentos de 1024 amostras
spectral = FeatureExtractor(
    {"power": partial(spectral_bands_power, bands=bands)},
    preprocessor=partial(
        spectral_preprocessor,
        fs=sfreq,
        nperseg=window_size,
        noverlap=0,
        f_min=8,
        f_max=16,
    ),
)
# Extrai as características espectrais em lote para todas as janelas
feature_table = extract_features(
    windows, {"spectral": spectral}, batch_size=64, n_jobs=1
).to_dataframe()
# Valida o número total de características (12 bandas * 8 canais = 96 colunas)
assert feature_table.shape == (len(y), len(class_names) * len(channel_names))
# Integra a potência multiplicando pela resolução em frequência (sfreq / window_size = 0.25 Hz) e aplica escala log
features = np.log(np.maximum(feature_table.to_numpy() * sfreq / window_size, 1e-30))
# Valida que todos os valores calculados são estritamente finitos
assert np.isfinite(features).all()

## 5. Ajustar em dois sujeitos e prever o terceiro
Cada sujeito é testado uma vez. Construa um novo pipeline em cada partição para que
nem o escalonamento nem o ajuste do classificador vejam o participante retido (*held-out*).
Hiperparâmetros são fixados aqui; a calibração de hiperparâmetros exigiria validação agrupada
dentro da partição de treino. A acurácia balanceada ao nível do acaso uniforme é 1/12,
desde que todas as doze classes ocorram na partição de teste, o que verificamos.
Cada partição externa contém 360 ensaios de treinamento de dois participantes e
180 ensaios de teste do terceiro. Um pipeline novo evita que o estado ajustado cruze
entre partições. O buffer de predições é preenchido nos índices originais das linhas;
``test_counts`` detecta tanto um ensaio de teste omitido quanto uma predição repetida.

Para calibrar a regularização ou a banda de frequência, divida os dois sujeitos de treino
novamente para validação interna antes de ajustar a configuração escolhida em ambos. Com
apenas dois sujeitos internos, essa calibração é instável; adicionar participantes é uma
extensão mais informativa do que uma grande grade de parâmetros.



In [ ]:
# Inicializa buffer para armazenar predições de teste alinhadas aos índices originais
predictions = np.full(len(y), -1, dtype=int)
# Vetor contador para verificar quantas vezes cada ensaio participou do teste
test_counts = np.zeros(len(y), dtype=int)
# Lista de dicionários para coletar resultados de cada sujeito retido
rows = []
# Executa validação cruzada deixando um participante fora em cada rodada (LOSO)
for train, test in LeaveOneGroupOut().split(features, y, groups):
    # Garante que os sujeitos de treino e teste são estritamente disjuntos (sem vazamento)
    assert set(groups[train]).isdisjoint(groups[test])
    # Garante que todas as doze classes estão presentes tanto no treino quanto no teste
    assert set(y[train]) == set(y[test]) == set(mapping.values())
    # Cria pipeline independente com escalonamento padronizado e regressão logística
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    # Ajusta o modelo estritamente nos dados dos dois participantes de treino
    model.fit(features[train], y[train])
    # Realiza predições para os ensaios do sujeito retido de teste
    predictions[test] = model.predict(features[test])
    # Incrementa contador de avaliações do conjunto de teste
    test_counts[test] += 1
    # Registra a acurácia balanceada obtida para o sujeito atual
    rows.append(
        {
            "subject": groups[test][0],
            "balanced_accuracy": balanced_accuracy_score(y[test], predictions[test]),
            "n_test_trials": len(test),
        }
    )
# Assegura que todas as rodadas foram processadas e cada ensaio foi avaliado exatamente uma vez
assert len(rows) == len(subjects)
assert np.all(test_counts == 1), "Every trial must be evaluated exactly once"
# Monta tabela com os resultados e imprime os dados
results = pd.DataFrame(rows)
print(results.to_string(index=False))
# Calcula a média e desvio padrão amostral das pontuações entre sujeitos
scores = results["balanced_accuracy"]
print(f"Subject mean +/- SD: {scores.mean():.3f} +/- {scores.std(ddof=1):.3f}")

## 6. Inspecionar resultados medidos
As barras exibem cada participante retido (*held-out*). A matriz de confusão agrega suas
predições e normaliza cada classe verdadeira. Nenhuma acurácia mínima é exigida:
um decodificador fraco é um resultado válido, enquanto uma divisão com vazamento é um erro metodológico.



In [ ]:
# Cria figura com dois subgráficos lado a lado
fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
# Plota o gráfico de barras com a acurácia balanceada de cada sujeito retido
axes[0].bar(results["subject"], scores)
# Adiciona linha tracejada com o nível de chance para 12 classes (1/12 ≈ 8.33%)
axes[0].axhline(1 / len(mapping), color="black", linestyle="--", label="Chance (1/12)")
axes[0].set(xlabel="Held-out subject", ylabel="Balanced accuracy", ylim=(0, 1))
axes[0].legend()
# Exibe matriz de confusão agrupada normalizada pelas classes verdadeiras
ConfusionMatrixDisplay.from_predictions(
    y,
    predictions,
    labels=list(mapping.values()),
    display_labels=class_names,
    normalize="true",
    include_values=False,
    colorbar=False,
    xticks_rotation=90,
    ax=axes[1],
)
axes[1].set_title("Attended frequency (Hz)")
fig.suptitle("Nakanishi2015 / nm000118: three-subject LOSO")
# Renderiza a figura
plt.show()

Três participantes mantêm este exemplo pequeno o suficiente para CI; eles não
estabelecem uma referência em nível populacional. Para estender a análise, adicione sujeitos reais
à consulta e mantenha a mesma avaliação disjunta por grupos. Para um decodificador neural,
reutilize essas janelas com ``EEGClassifier`` e reserve sujeitos de validação dentro de cada partição de treino.



In [ ]:
# Leitura dos dois gráficos
# ---------------------
# A média por sujeito pondera as pessoas igualmente. Seu DP amostral descreve a variação
# entre essas três pessoas; não é um intervalo de confiança para uma população.
# A matriz de confusão normalizada responde, para cada frequência verdadeira, quais
# frequências recebem suas predições. Suas linhas somam um, portanto uma célula mais escura
# significa uma fração maior daquela classe verdadeira, não mais ensaios na coorte.
#
# Inspecione confusões entre frequências próximas antes de propor uma linha de base
# espectral mais refinada. Congele essa proposta antes de avaliar sujeitos adicionais retidos.
# Os participantes já inspecionados tornam-se dados de desenvolvimento para esse próximo experimento.